# Data check: `MORTALITAT_GLOBAL`

This notebook profiles the global mortality dataset, checks its variable structure, and highlights common data quality issues such as missing values, duplicates, and unexpected category patterns.

**Import the analysis dependencies and set display options for wide summary tables.**

Output: no analytical result yet; this cell only configures the notebook session.

In [3]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

**Load the source file with the correct delimiter and standardize the metric column for downstream checks.**

Output: confirms the file path, dataset shape, column names, and shows the first rows.

In [4]:
csv_path = Path("MORTALITAT_GLOBAL.csv")
df = pd.read_csv(csv_path, sep=",")
df["TAXA"] = pd.to_numeric(df["TAXA"], errors="coerce")

print(f"File: {csv_path.resolve()}")
print(f"Rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'MORTALITAT_GLOBAL.csv'

**Profile each variable by data type, missingness, and cardinality.**

Output: one row per column, useful for spotting sparse fields or incorrectly parsed variables.

In [ ]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "unique_values": df.nunique(dropna=True)
})
summary

**Inspect the categorical distributions of the main descriptive variables.**

Output: frequency tables that reveal dominant categories, unexpected labels, or imbalance across groups.

In [ ]:
for column in ["CAUSA", "SEXE", "Territori", "EDAT"]:
    print(f"\n{column}")
    display(df[column].value_counts(dropna=False).rename("count").to_frame())

**Detect exact row-level duplicates across the full record.**

Output: duplicate count and a sample of repeated rows for direct inspection.

In [ ]:
duplicate_mask = df.duplicated()
duplicate_rows = df[duplicate_mask].sort_values(df.columns.tolist()).reset_index(drop=True)

print(f"Exact duplicate rows: {duplicate_mask.sum()}")
duplicate_rows.head(20)

**Check whether the analytical key is unique once `TAXA` is excluded.**

Output: repeated key combinations that may indicate duplicated observations or an incomplete key definition.

In [ ]:
key_columns = ["CAUSA", "SEXE", "Territori", "EDAT"]
duplicate_keys = (
    df.groupby(key_columns, dropna=False)
      .size()
      .reset_index(name="n_rows")
      .query("n_rows > 1")
      .sort_values("n_rows", ascending=False)
)

print(f"Repeated variable combinations: {len(duplicate_keys)}")
duplicate_keys.head(20)

**Summarize the distribution of the response variable before grouped comparisons.**

Output: standard descriptive statistics for `TAXA`, including spread and extreme values.

In [ ]:
taxa_summary = df["TAXA"].describe()
taxa_summary

**Compare `TAXA` across sex, territory, and age group.**

Output: grouped summary tables that surface broad differences in mortality rates across the main analytical dimensions.

In [ ]:
display(
    df.groupby("SEXE", dropna=False)["TAXA"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_index()
)

display(
    df.groupby("Territori", dropna=False)["TAXA"]
      .agg(["count", "mean", "median", "min", "max"])
      .sort_values("mean", ascending=False)
)

display(
    df.groupby("EDAT", dropna=False)["TAXA"]
      .agg(["count", "mean", "median", "min", "max"])
)


**Rank causes by aggregate `TAXA` to identify the most influential groups.**

Output: the leading cause categories by mean `TAXA`, with supporting count, median, and maximum values.

In [ ]:
top_causes = (
    df.groupby("CAUSA", dropna=False)["TAXA"]
      .agg(["count", "mean", "median", "max"])
      .sort_values("mean", ascending=False)
)
top_causes.head(15)

**Reshape the data into a matrix for side-by-side comparison across territory and sex.**

Output: a compact cross-tab where each row is a cause and each column is a territory and sex combination.

In [ ]:
pivot_table = pd.pivot_table(
    df,
    values="TAXA",
    index=["CAUSA", "EDAT"],
    columns=["Territori", "SEXE"],
    aggfunc="mean"
)
pivot_table.head(20)

**Consolidate the core data-quality indicators into a final checkpoint.**

Output: a compact diagnostic series summarizing missingness, duplicates, invalid values, and zeros.

In [ ]:
quality_checks = {
    "missing_values_total": int(df.isna().sum().sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "negative_taxa": int((df["TAXA"] < 0).sum()),
    "zero_taxa": int((df["TAXA"] == 0).sum()),
    "non_numeric_taxa_after_conversion": int(df["TAXA"].isna().sum())
}

pd.Series(quality_checks, name="result")